# Μάθημα 18: Ασφάλεια AI πράκτορων με Κρυπτογραφικές Αποδείξεις

## Πρακτικό Τετράδιο

Αυτό το τετράδιο διατρέχει τέσσερις ασκήσεις:

1. **Υπογράψτε την πρώτη σας απόδειξη** για μια κλήση εργαλείου πράκτορα και επαληθεύστε την.
2. **Παραποιήστε την απόδειξη** και παρακολουθήστε την αποτυχία επαλήθευσης.
3. **Δημιουργήστε μια αλυσίδα τριών αποδείξεων** και επιβεβαιώστε την ακεραιότητα της αλυσίδας.
4. **Περιτυλίξτε μια κλήση εργαλείου από το Microsoft Agent Framework** ώστε κάθε ενέργεια να εκπέμπει μια απόδειξη.

Όλες οι κρυπτογραφικές πράξεις εισάγονται από καλά συντηρημένες βιβλιοθήκες (`pynacl` για Ed25519, `jcs` για canonical JSON κατά RFC 8785, `hashlib` από την τυπική βιβλιοθήκη Python για SHA-256). Η λογική της απόδειξης είναι απλό Python που μπορείτε να διαβάσετε και να τροποποιήσετε.

Εκτελέστε τα κελιά με τη σειρά τους. Κάθε ενότητα είναι σύντομη και αυτοτελής.


## Ρύθμιση

Εγκαταστήστε τις δύο εξαρτήσεις. Και οι δύο έχουν άδειες χρήσης permissive (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Βοηθητικά Εργαλεία

Αυτά τα δύο βοηθητικά χειρίζονται τον κωδικοποιητή base64url (χωρίς padding) και το κατακερματισμό SHA-256 αυθαίρετων αντικειμένων. Διατηρούν το υπόλοιπο του σημειωματάριου εστιασμένο στην ίδια τη λογική απόδειξης.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Υπογράψτε την πρώτη σας απόδειξη

Φανταστείτε ότι ο πράκτοράς μας για την **Contoso Travel** μόλις αναζήτησε πτήσεις από το Σίδνεϊ προς το Λος Άντζελες για έναν πελάτη. Θέλουμε να καταγράψουμε αυτήν την κλήση εργαλείου ως υπογεγραμμένη απόδειξη ώστε ένας μελλοντικός ελεγκτής να μπορεί να την επαληθεύσει χωρίς να χρειάζεται να μας εμπιστευτεί.

### Step 1.1: Δημιουργία κλειδιού υπογραφής

Στην παραγωγή, το κλειδί υπογραφής του πράκτορα θα βρισκόταν σε μια μονάδα ασφάλειας υλικού (HSM), το Azure Key Vault, ή σε ένα παρόμοιο προστατευμένο αποθετήριο. Για αυτό το μάθημα δημιουργούμε ένα νέο κλειδί στη μνήμη.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Βήμα 1.2: Δημιουργία του φορτίου απόδειξης

Το φορτίο περιλαμβάνει ό,τι θέλουμε να πιστοποιεί η απόδειξη: ποιος ενήργησε, ποιο εργαλείο, με ποια επιχειρήματα, τι επέστρεψε, υπό ποια πολιτική και πότε. Κάνουμε κατακερματισμό των επιχειρημάτων και του αποτελέσματος αντί να τα συμπεριλάβουμε απευθείας, ώστε η απόδειξη να μην αποκαλύπτει ευαίσθητο περιεχόμενο.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Βήμα 1.3: Υπογράψτε και συναρμολογήστε την απόδειξη

Τρία βήματα:

1. Κανoνικοποιήστε το περιεχόμενο χρησιμοποιώντας JCS ώστε δύο υλοποιήσεις που παράγουν την ίδια λογική απόδειξη να παράγουν ακριβώς τα ίδια bytes.
2. Κάντε κατακερματισμό (hash) των κανονικοποιημένων bytes με SHA-256.
3. Υπογράψτε τον κατακερματισμό με το ιδιωτικό κλειδί Ed25519.

Η υπογραφή στη συνέχεια επισυνάπτεται στο αρχικό περιεχόμενο για να παραχθεί η τελική απόδειξη.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Βήμα 1.4: Επαλήθευση της απόδειξης

Η επαλήθευση αντιστρέφει τη διαδικασία. Αφαιρούμε την υπογραφή, επανυπολογίζουμε τον κανονικό κατακερματισμό και ελέγχουμε την υπογραφή σε σχέση με το δημόσιο κλειδί στην απόδειξη.

Ένας ελεγκτής που πραγματοποιεί αυτήν την επαλήθευση δεν χρειάζεται τίποτα από εμάς εκτός από την ίδια την απόδειξη. Κανένα υπηρεσία για κλήση, κανέναν κατάλογο κλειδιών για ερώτηση, καμία απαιτούμενη εμπιστοσύνη.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Θα πρέπει να δείτε `Receipt is valid: True`. Ο πράκτορας έχει παράγει την πρώτη του κρυπτογραφικά υπογεγραμμένη εγγραφή ελέγχου.


## Ενότητα 2: Παρεμβολή στην απόδειξη

Ολόκληρος ο σκοπός των αποδείξεων είναι να είναι αντιληπτές οι παρεμβάσεις σε αυτές. Ας το αποδείξουμε.

Θα τροποποιήσουμε ακριβώς έναν χαρακτήρα της απόδειξης και θα παρακολουθήσουμε την αποτυχία της επαλήθευσης.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Τι μόλις συνέβη;

Όταν αλλάξαμε το `policy_id`, τα κανονικά bytes άλλαξαν. Το SHA-256 hash αυτών των bytes άλλαξε. Η υπογραφή (η οποία ήταν πάνω στο αρχικό hash) δεν ταιριάζει πλέον με το νέο hash. Ο έλεγχος επαλήθευσης επιστρέφει σωστά `False`.

Δεν υπάρχει τρόπος να τροποποιηθεί οποιοδήποτε πεδίο της απόδειξης και να εξακολουθεί να επαληθεύεται, εκτός αν ο επιτιθέμενος έχει το ιδιωτικό κλειδί. Εφόσον το ιδιωτικό κλειδί βρίσκεται σε θησαυροφυλάκιο κλειδιών και το δημόσιο κλειδί έχει δημοσιευτεί, η παραβίαση είναι αδύνατο να κρυφτεί.

Δοκιμάστε το μόνοι σας: τροποποιήστε το `tool_name` ή το `agent_id` ή το `timestamp` στο κελί παραπάνω και τρέξτε ξανά. Κάθε αλλαγή παράγει μια άκυρη απόδειξη.


## Section 3: Αλυσίδα αποδείξεων μαζί

Μια μεμονωμένη απόδειξη προστατεύει μια ενέργεια. Οι περισσότεροι πράκτορες εκτελούν πολλές ενέργειες. Για να κάνουμε ολόκληρη τη σειρά ανθεκτική σε παραποιήσεις, συνδέουμε κάθε απόδειξη με την προηγούμενη συμπεριλαμβάνοντας το hash της προηγούμενης απόδειξης στο payload της νέας απόδειξης.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Εάν κάποιος αφαιρέσει ή αλλάξει τη σειρά μιας απόδειξης, η αλυσίδα σπάει ακριβώς σε εκείνο το σημείο. Η επαλήθευση οποιασδήποτε μεταγενέστερης απόδειξης αποτυγχάνει επειδή το `previous_receipt_hash` δεν ταιριάζει πλέον με το πραγματικό hash του προκατόχου της.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Τώρα σπάστε την αλυσίδα παραποιώντας την απόδειξη στη μέση και επανελέγξτε. Η παραποιημένη απόδειξη αποτυγχάνει τον έλεγχο της υπογραφής της, ΚΑΙ η επόμενη απόδειξη αποτυγχάνει τον έλεγχο της αλυσίδας (επειδή το `previous_receipt_hash` της δεν ταιριάζει πλέον με το τροποποιημένο hash της απόδειξης στη μέση).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Η απόδειξη 0 εξακολουθεί να επαληθεύεται (δεν τροποποιήθηκε και δεν έχει πρόγονο στον οποίο να εξαρτάται). Η απόδειξη 1 αποτυγχάνει στον έλεγχο της υπογραφής της επειδή αλλάξαμε το `tool_args_hash`. Η απόδειξη 2 αποτυγχάνει στον έλεγχο του συνδέσμου της αλυσίδας γιατί το `previous_receipt_hash` της υπολογίστηκε σε σχέση με την αρχική (τώρα τροποποιημένη) απόδειξη 1.

Ακόμα και αν ένας επιτιθέμενος επαναϋπογράψει την τροποποιημένη απόδειξη 1 (κάτι που δεν μπορεί να κάνει χωρίς το ιδιωτικό κλειδί), η ασυμφωνία του συνδέσμου της αλυσίδας στην απόδειξη 2 θα αποκάλυπτε ακόμα την παραβίαση. Για να κρύψει την αλλαγή, ο επιτιθέμενος θα έπρεπε να επαναϋπογράψει κάθε απόδειξη από το σημείο της τροποποίησης και έπειτα, κάτι που απαιτεί την κατοχή του ιδιωτικού κλειδιού.


## Section 4: Εμφάνιση κλήσης εργαλείου πράκτορα με υπογραφή απόδειξης

Σε μια πραγματική υλοποίηση, δεν θέλετε κάθε δημιουργό πράκτορα να θυμάται να καλεί το `make_receipt`. Θέλετε η υπογραφή απόδειξης να είναι αυτόματη για κάθε εκτέλεση εργαλείου.

Εδώ είναι το απλούστερο μοτίβο: μια κλάση περιτύλιξης που παίρνει οποιαδήποτε καλούμενη λειτουργία εργαλείου και επιστρέφει μια έκδοση που εκπέμπει απόδειξη. Αυτό προσαρμόζεται σε οποιοδήποτε πλαίσιο πράκτορα, συμπεριλαμβανομένου του Microsoft Agent Framework (`agent_framework.azure`).

Αν δεν έχετε ρυθμίσει ένα έργο Azure AI Foundry, ο τοπικός προσομοιωτής παρακάτω εξακολουθεί να επιδεικνύει το μοτίβο.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Ενσωμάτωση με το Πλαίσιο Πράκτορα της Microsoft

Ο περιτυλιγμένος `ReceiptedTool` παραπάνω είναι ανεξάρτητος από πλαίσιο. Για να τον χρησιμοποιήσετε μέσα σε έναν πράκτορα που έχει κατασκευαστεί με το Πλαίσιο Πράκτορα της Microsoft, καταχωρήστε τη περιτυλιγμένη λειτουργία ως εργαλείο. Ένα προσχέδιο (θα αντικαταστήσετε το ψεύτικο με μια πραγματική καταχώρηση εργαλείου Azure AI Foundry):

```python
# Ψευδοκώδικας που δείχνει το σχήμα ολοκλήρωσης.
# από agent_framework.azure εισάγετε τον AzureAIProjectAgentProvider
# από azure.identity εισάγετε το AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     οδηγίες="Είστε πράκτορας ταξιδιών Contoso ..."
#     εργαλεία=[receipted_lookup],   # το περιτυλιγμένο εργαλείο, όχι η ακατέργαστη λειτουργία
# )
# απάντηση = agent.run("Βρείτε πτήσεις από το Σίδνεϊ στο Λος Άντζελες τον Ιούνιο.")
#
# # Μετά την εκτέλεση, κάθε κλήση εργαλείου που έκανε ο πράκτορας έχει μια υπογεγραμμένη απόδειξη:
# audit_chain = receipted_lookup.receipts
```

Το πλαίσιο πράκτορα δεν χρειάζεται να γνωρίζει τίποτα για τις αποδείξεις. Η υπογραφή της απόδειξης είναι περιτυλιγμένη γύρω από το εργαλείο, όχι ενσωματωμένη στο πλαίσιο. Έτσι προσθέτετε προέλευση στον υπάρχοντα κώδικα πράκτορα χωρίς να ξαναγράψετε τον πράκτορα.


## Ανασκόπηση και πρόκληση επέκτασης

Έχετε:

- Δημιουργήσει ένα ζεύγος κλειδιών Ed25519.  
- Δημιουργήσει και υπογράψει μία απόδειξη για κλήση εργαλείου πράκτορα.  
- Επαληθεύσει την απόδειξη εκτός σύνδεσης χρησιμοποιώντας μόνο το δημόσιο κλειδί.  
- Παραποιήσει μια απόδειξη και παρατηρήσει αποτυχία επαλήθευσης.  
- Δημιουργήσει μια αλυσιδωτή ακολουθία τριών αποδείξεων μέσω κατακερματισμού.  
- Παραποιήσει το μέσο της αλυσίδας και παρατηρήσει τόσο αποτυχία υπογραφής όσο και αποτυχία σύνδεσης αλυσίδας.  
- Τυλίξει μια συνάρτηση εργαλείου με αυτόματο υπογραφή απόδειξης.

**Πρόκληση επέκτασης.** Επεκτείνετε το σχήμα της απόδειξης με ένα πεδίο `request_id` (ένα UUID για κατανεμημένη ιχνηλασιμότητα). Ενημερώστε το `make_receipt` ώστε να το περιλαμβάνει, και επιβεβαιώστε ότι οι αποδείξεις επαληθεύονται κανονικά από άκρο σε άκρο. Στη συνέχεια τροποποιήστε το πεδίο μετά την υπογραφή και επιβεβαιώστε ότι η επαλήθευση αποτυγχάνει. Αυτό σας αναγκάζει να καταλάβετε πώς κάθε byte της κανονικής κωδικοποίησης συμβάλλει στην υπογραφή.

**Σημαντικό όριο.** Οι αποδείξεις αποδεικνύουν τρία πράγματα και μόνο τρία: ανάθεση (αυτό το κλειδί υπέγραψε αυτό το περιεχόμενο), ακεραιότητα (το περιεχόμενο δεν έχει αλλάξει από την υπογραφή), και σειρά (αυτή η απόδειξη ήρθε μετά από εκείνη την απόδειξη). ΔΕΝ αποδεικνύουν ότι η ενέργεια του πράκτορα ήταν σωστή, ότι η πολιτική με το όνομα στο `policy_id` αξιολογήθηκε όντως, ή ότι ο πράκτορας ακολούθησε κάθε κανόνα. Οι αποδείξεις είναι η βάση. Η διακυβέρνηση είναι το σύστημα που χτίζετε από πάνω.

Διαβάστε ξανά το README του μαθήματος με αυτό το όριο κατά νου. Το πιο κοινό λάθος που κάνουν οι ομάδες με τις αποδείξεις είναι να υποθέτουν ότι «έχουμε αποδείξεις» σημαίνει «ελεγχόμαστε». Δεν ισχύει. Οι αποδείξεις καθιστούν τη συμπεριφορά του πράκτορα ελεγχόμενη. Δεν την καθιστούν σωστή.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
